In [ ]:
# peft :

# 0000000000000000000-------
# 1. Selective : intrinsic dimensionality : 일부만 쪼개서 업데이트
# 2. Additive :  00000000000000+++0000000000 , token sequence 앞에 가상토큰, soft prompt
# 3. Reparametrization        SVD   LoRA    Low Rank Adapter   -> QLoRA

In [ ]:
import os
import json
import time
import numpy as np
import matplotlib

import torch
import torch.nn as nn

from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

from peft import get_peft_model, LoraConfig, TaskType

2026-03-18 10:52:34.179423: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773798754.197789  631275 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773798754.203870  631275 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773798754.218789  631275 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773798754.218810  631275 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773798754.218812  631275 computation_placer.cc:177] computation placer alr

In [ ]:
class SimpleAdapter:

    def __init__(self, input_dim, bottleneck_dim):
        self.input_dim = input_dim
        self.bottleneck_dim = bottleneck_dim
        self.W_down = np.random.randn(input_dim, bottleneck_dim)    # nn.Linear(input_dim, bottleneck_dim)  : input_dim * bottleneck_dim
        self.W_up = np.random.randn(bottleneck_dim, input_dim)     # siamness architecture                  : bottleneck_dim * input_dim

    def forward(self, x):
        down = x @ self.W_down
        activated = np.maximum(down, 0)
        up = activated @ self.W_up

        return x + up

    def num_params(self):
        return self.input_dim * self.bottleneck_dim * 2

In [ ]:
hidden_dim = 768
bottleneck_sizes = [8, 16, 32, 64, 128, 256]

test_input = np.random.randn(32, hidden_dim)

In [ ]:
full_params = hidden_dim * hidden_dim
for bn in bottleneck_sizes:
    adapter = SimpleAdapter(hidden_dim, bn)
    n_param = adapter.num_params()
    compression = full_params / n_param
    print(f"{bn} : {n_param} / {full_params}, {compression}")

8 : 12288 / 589824, 48.0
16 : 24576 / 589824, 24.0
32 : 49152 / 589824, 12.0
64 : 98304 / 589824, 6.0
128 : 196608 / 589824, 3.0
256 : 393216 / 589824, 1.5


In [ ]:
# Selective

# Layer freeze, parameter update .


In [ ]:
# Reparametrization
# LoRA : Low Rank Adapter

model weight : W
W -> W'
W' = W + delta_W

W + delta_W

delta_W : A X B
         (d,r) X (r, d) -> (d, d)
W : d x d


SyntaxError: unterminated string literal (detected at line 5) (37060016.py, line 5)

In [ ]:
(100, 100) -> (100, 2) x (2, 100)  # 1만개 -> 400개

In [ ]:
(100, 50) x (50, 100)


In [ ]:
# SVD
# W = U, sigma, V  = UxSigmaxV^T
# Sigma : 0 , 1,2,3,
# Low Rank : k
# Rank 들이 W의 정보를 담고있다.
# Rank : 100 : 100  0.01
# top 10 rank

In [ ]:
np.linag.svd

In [ ]:
d  = 100
W_delta = np.random.randn(d, d)* 0.01

U, S, Vt = np.linalg.svd(W_delta, full_matrices=False)

In [ ]:
U.shape, S.shape, Vt.shape

((100, 100), (100,), (100, 100))

In [ ]:
for i in range(20):
    print(f"S[{i}] : {S[i]}")

S[0] : 0.19753130509857367
S[1] : 0.19019216970855485
S[2] : 0.18516873776801065
S[3] : 0.18164569967207536
S[4] : 0.17539179034775815
S[5] : 0.17426712791958124
S[6] : 0.17221503584047876
S[7] : 0.169704569628563
S[8] : 0.1657028443005344
S[9] : 0.16249198514936536
S[10] : 0.16094645800640753
S[11] : 0.15818682237905607
S[12] : 0.1556066890978529
S[13] : 0.15281699681742833
S[14] : 0.15059706642211593
S[15] : 0.14991013025710856
S[16] : 0.14610250975304864
S[17] : 0.14107490413457502
S[18] : 0.14067965468137442
S[19] : 0.13959769103321965


In [ ]:
def low_rank_approx(W, rank):

    U_k, S_k, Vt_k = np.linalg.svd(W, full_matrices=False)
    W_approx = U_k[:,:rank] @ np.diag(S_k[:rank]) @ Vt_k[:rank, :]

    m, n = W.shape
    compressed_params = m*rank + rank + n*rank

    relative_error = np.linalg.norm(W-W_approx) / np.linalg.norm(W)

    return W_approx, compressed_params, relative_error


In [ ]:
ranks = [1, 2, 4, 8, 16, 32, 64, 100]
original_params = d*d

for r in ranks:
    _, compressed_params, relative_error = low_rank_approx(W_delta, r)
    compression_pct = (1-compressed_params/original_params) * 100

    print(f"{r} : {original_params} | {compressed_params} | {compression_pct} % | {relative_error}")

1 : 10000 | 201 | 97.99 % | 0.9804169940924635
2 : 10000 | 402 | 95.98 % | 0.9619060772446147
4 : 10000 | 804 | 91.96 % | 0.9264922028893701
8 : 10000 | 1608 | 83.91999999999999 % | 0.85995491430186
16 : 10000 | 3216 | 67.84 % | 0.7370282516861709
32 : 10000 | 6432 | 35.68 % | 0.5247184400127328
64 : 10000 | 12864 | -28.64 % | 0.19553166809077144
100 : 10000 | 20100 | -100.99999999999997 % | 2.2464617067611776e-15


In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
config = LoraConfig(
    r = 8,
    lora_alpha =32,
    target_modules = ["query", "value"],
    task_type = TaskType.SEQ_CLS,
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=2)
peft_model = get_peft_model(model, config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/oncreative/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [ ]:
peft_model.print_trainable_parameters()

trainable params: 296,450 || all params: 178,151,428 || trainable%: 0.1664


In [ ]:
config = LoraConfig(
    r = 16,
    lora_alpha =32,
    target_modules = ["query", "value"],
    task_type = TaskType.SEQ_CLS,
)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=2)
peft_model = get_peft_model(model, config)

peft_model.print_trainable_parameters()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 591,362 || all params: 178,446,340 || trainable%: 0.3314


In [ ]:
class LoraLinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha, dropout =  0.8):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.linear.weight.requires_grad = False
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha/rank
        self.A = nn.Parameter(torch.randn(in_features, rank) * (1.0 / rank ** 0.5))
        self.B = nn.Parameter(torch.zeros(rank, out_features))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        base_out = self.linear(x)
        lora_out = self.scaling * (self.dropout(x) @ self.A @ self.B)

        return base_out + lora_out

def analyze_config(name, r, alpha, d_in=768, d_out=768, dropout = 0.0, n_samples=100):
    layer = LoraLinear(d_in, d_out, r, alpha, dropout)

    with torch.no_grad():
        layer.B.copy_(torch.randn(r, d_out) * 0.01)

    layer.eval()
    outputs = []
    lora_deltas = []

    for _ in range(n_samples):
        x = torch.randn(1, d_in)
        with torch.no_grad():
            full_out = layer(x)
            base_out = layer.linear(x)
            delta = full_out - base_out
            outputs.append(full_out)
            lora_deltas.append(delta)

    all_deltas = torch.cat(lora_deltas)
    all_outputs = torch.cat(outputs)

    total_params = sum(p.numel() for p in layer.parameters())
    trainable_params = sum(p.numel() for p in layer.parameters() if p.requires_grad)

    return {
            "name" : name,
            "r" : r,
            "alpha" : alpha,
            "scaling" : alpha / r,
            "dropout" : dropout,
            "trainable_params" : trainable_params,
            "total_params" : total_params,
            "output_mean" : all_outputs.abs().mean().item(),
            "output_std" : all_outputs.std().item(),
            "delta_mean" : all_deltas.abs().mean().item(),
            "delta_std" : all_deltas.std().item(),
        }

In [ ]:
configs = [
    ("minimal", 4, 8, 0.0), ("base", 8, 32, 0.05), ("aggressive", 16, 32, 0.1), ("max", 64, 128, 0.1)]

In [ ]:
results = []
for name, r, alpha, dropout in configs:
    result = analyze_config(name=name, r=r, alpha=alpha, d_in=768, d_out=768, dropout = dropout, n_samples=100)
    results.append(result)

In [ ]:
print(f" name | r | alpha | scaling | dropout | train param | total param |  output_mean | output_std")
for res in results:
    print(f" {res['name']} | {res['r']} | {res['alpha']} | {res['scaling']} | {res['dropout']} | {res['trainable_params']} | {res['total_params']} | {res['output_mean']} |{res['output_std']} ")

 name | r | alpha | scaling | dropout | train param | total param |  output_mean | output_std
 minimal | 4 | 8 | 2.0 | 0.0 | 6144 | 595968 | 0.6224997639656067 |0.7879238724708557 
 base | 8 | 32 | 4.0 | 0.05 | 12288 | 602112 | 1.0042643547058105 |1.2859585285186768 
 aggressive | 16 | 32 | 2.0 | 0.1 | 24576 | 614400 | 0.6330543756484985 |0.7972918152809143 
 max | 64 | 128 | 2.0 | 0.1 | 98304 | 688128 | 0.6357450485229492 |0.7966198325157166 


In [ ]:
W_delta : SVD
W + W_delta

In [ ]:
tunneling : ngrok, cloudflare, ....

ngrok :
cloudflare :